In [ ]:
import os
import numpy as np
import scipy as sp
from src.data import DataLoader
from src.timescales import Timescales

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
dataset = 'HCP-YA'
data_loader = DataLoader(dataset=dataset)

# fMRI data

In [ ]:
subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids.txt'), dtype=str)
# subject_ids = subject_ids[:10]
subject_ids

In [ ]:
# Register 3T rest scans
rest_scans = data_loader.register_3t_rest_scans(
    subject_ids=subject_ids,
    atlas='Schaefer4007',
)

# Register 3T task scans
task_scans = data_loader.register_3t_task_scans(
    subject_ids=subject_ids,
    atlas='Schaefer4007',
)

# Register 7T rest scans
task_scans = data_loader.register_7t_rest_scans(
    subject_ids=subject_ids,
    atlas='Schaefer4007',
)

# Register 7T movie scans
task_scans = data_loader.register_7t_movie_scans(
    subject_ids=subject_ids,
    atlas='Schaefer4007',
)

In [ ]:
_ = data_loader.validate_scan_registry()

In [ ]:
data_loader.export_scan_registry('scan_registry_fmri.csv')
scan_registry = data_loader.list_registered_scans()
scan_registry.head()

In [ ]:
subject_ids = scan_registry['subject_id'].unique()
n_subs = len(subject_ids)
print(n_subs)

In [ ]:
# Load the data
fmri_data = data_loader.load_batch(
    nuisance_regression=True,
    verbose=True
)

print(f"Loaded {len(fmri_data)} scans")
print(fmri_data.keys())

## Process fMRI timescales

In [ ]:
timescales_processor = Timescales(dataset=dataset, timeseries_data=fmri_data, verbose=True)

In [ ]:
timescale_measures = [
    'zero_crossing',
    'auc_zero_crossing'
]

In [ ]:
timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, verbose=True)
for timescale_measure in timescale_measures:
    timescales_processor.compute_timescale_measures(method=timescale_measure, verbose=True)

In [ ]:
timescales_processor.regress_task_structure()

In [ ]:
timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, verbose=True)
for timescale_measure in timescale_measures:
    timescales_processor.compute_timescale_measures(method=timescale_measure, suffix='resid', verbose=True)